# 基准锚定模型：实际交易日自 2026-08-03 起的固定窗口诊断

本表澄清“8月3日以来”的日期含义：按 **trade_date ≥ 2026-08-03**，不是 signal_date。原 card 的信号窗口结果保留不动。只使用已完成 Qlib Recorder `52f40778cbe64ca6a5864da356b960d0` 中选定的 `price` 模型、既有诊断输入和连续持仓路径。

固定比较登记过的 annual_refit / frozen2022_parameters 两种模式、10/25bp 两档费用、持续持仓 / 独立现金起点。不会重新训练、选参数或选择模式。数据终点为 **2026-09-25 OPEN**；不能与其他来源或截至 9月30日 CLOSE 的旧结果直接比较。

In [1]:
import os,sys,json,pickle,hashlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
ROOT=Path('/Users/rziiiii/dev/workspace-worktrees/benchmark-anchor-utility-20261002')
ROOT=ROOT if ROOT.is_dir() else Path('/Users/rziiiii/dev/workspace')
sys.path.insert(0,str(ROOT))
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from IPython.display import display,Markdown
from src.workflow.benchmark_anchor_utility import replay_models
from src.workflow.stable_excess_integration import benchmark_path
from src.analysis.integrated_allocation import paired_metrics
summary=json.loads((ROOT/'research/benchmark_anchor_utility_20261002/summary.json').read_text())
assert summary['recorder_id']=='52f40778cbe64ca6a5864da356b960d0'
for name in ('src/data/benchmark_anchor_utility.py','src/model/benchmark_anchor_utility.py','src/workflow/benchmark_anchor_utility.py'):
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==summary['source_sha256'][name]
qlib.init(provider_uri=str(ROOT/summary['provider_uri']),region='us',kernels=1)
with R.uri_context(summary['recorder_uri']):
    rec=R.get_recorder(experiment_name=summary['experiment'],recorder_id=summary['recorder_id'])
    assert rec.status=='FINISHED'
    frames=rec.load_object('inputs/diagnostic_frames.pkl')
    fitted=rec.load_object('models/selected_annual_refits.pkl')
    saved=rec.load_object('analysis/portfolio_replays.pkl')
    original_windows=rec.load_object('analysis/window_metrics.pkl')
chosen=summary['selected']['variant'];assert chosen=='price'
START=pd.Timestamp('2026-08-03');ASOF='2026-09-25'
print({'source_recorder':rec.id,'status':rec.status,'policy':chosen,'original_signal_windows_unchanged':True,'model_fits':False})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[99896:MainThread](2026-10-03 00:24:30,392) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[99896:MainThread](2026-10-03 00:24:30,394) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[99896:MainThread](2026-10-03 00:24:30,394) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-worktrees/benchmark-anchor-utility-20261002/data/qlib/published_futu_affine_20260925_v1')}


{'source_recorder': '52f40778cbe64ca6a5864da356b960d0', 'status': 'FINISHED', 'policy': 'price', 'original_signal_windows_unchanged': True, 'model_fits': False}


## 对齐实际持有区间

持续持仓结果直接切片 Recorder 中自2023年起的既有模拟完整路径（不是用户真实账户），保留其真实期初仓位。独立现金结果从相同首个交易日用**原样保存的模型**重算，并为策略和 SOXX 同时计入新入场费。新表单独保存；没有覆盖已登记的核心路径或其信号日期窗口。

In [2]:
metrics=[];new_paths={}
for mode in ('annual_refit','frozen2022_parameters'):
    frame=frames[mode]
    selected=frame.loc[pd.to_datetime(frame.trade_date).ge(START)].copy()
    assert not selected.empty
    assert pd.Timestamp(selected.trade_date.iloc[0])==START
    assert str(pd.Timestamp(selected.exit_date.iloc[-1]).date())==ASOF
    policy_name=chosen+'_'+mode
    for cost in (10,25):
        existing=saved.loc[saved.variant.eq(policy_name)&saved.seed.eq('ensemble')&saved.segment.eq('reused_diagnostic')&saved.cost_bps.eq(cost)].sort_values('signal_date')
        reference=saved.loc[saved.variant.eq('SOXX_buy_hold')&saved.seed.eq('benchmark')&saved.segment.eq('reused_diagnostic')&saved.cost_bps.eq(cost)].sort_values('signal_date')
        carried=existing.loc[pd.to_datetime(existing.trade_date).ge(START)].copy()
        carried_ref=reference.loc[pd.to_datetime(reference.trade_date).ge(START)].copy()
        assert len(carried)==len(selected)==len(carried_ref)
        fresh=replay_models(selected,fitted[chosen],variant=chosen,cost_bps=cost,frozen_year=2023 if mode=='frozen2022_parameters' else None)
        fresh_ref=benchmark_path(selected,cost_bps=cost)
        for origin,path,benchmark in [('carried_book',carried,carried_ref),('cash_at_first_trade',fresh,fresh_ref)]:
            assert pd.to_datetime(path.signal_date).tolist()==pd.to_datetime(selected.signal_date).tolist()
            assert pd.to_datetime(path.trade_date).tolist()==pd.to_datetime(benchmark.trade_date).tolist()
            measure=paired_metrics(path,benchmark)
            metrics.append({'variant':chosen,'mode':mode,'cost_bps':cost,'origin':origin,'filter':'trade_date >= 2026-08-03','first_signal':str(pd.Timestamp(path.signal_date.iloc[0]).date()),'first_trade':str(pd.Timestamp(path.trade_date.iloc[0]).date()),'last_exit':str(pd.Timestamp(path.exit_date.iloc[-1]).date()),'endpoint_mark':'OPEN','first_pretrade_soxl':float(path.pretrade_soxl_weight.iloc[0]),'first_pretrade_soxx':float(path.pretrade_soxx_weight.iloc[0]),'first_pretrade_cash':float(path.pretrade_cash_weight.iloc[0]),**measure})
            new_paths[(mode,cost,origin)]={'strategy':path,'benchmark':benchmark}
table=pd.DataFrame(metrics)
display(table[['mode','cost_bps','origin','first_signal','first_trade','last_exit','sessions','return_pct','benchmark_return_pct','relative_return_pct','max_drawdown_pct','benchmark_max_drawdown_pct']].round(5))
print('Original SIGNAL window:',original_windows[['start_signal','last_exit','sessions']].drop_duplicates().to_dict('records'))
print('Clarified TRADE window:',table[['first_signal','first_trade','last_exit','sessions']].drop_duplicates().to_dict('records'))

,mode,cost_bps,origin,first_signal,first_trade,last_exit,sessions,return_pct,benchmark_return_pct,relative_return_pct,max_drawdown_pct,benchmark_max_drawdown_pct
0,annual_refit,10,carried_book,2026-07-31,2026-08-03,2026-09-25,38,17.19700,15.84322,1.16863,-12.25972,-10.98365
1,annual_refit,10,cash_at_first_trade,2026-07-31,2026-08-03,2026-09-25,38,12.84629,15.72750,-2.48965,-12.25971,-10.98365
2,annual_refit,25,carried_book,2026-07-31,2026-08-03,2026-09-25,38,16.99770,15.84322,0.99658,-12.30823,-10.98365
3,annual_refit,25,cash_at_first_trade,2026-07-31,2026-08-03,2026-09-25,38,12.53554,15.55434,-2.61245,-12.30821,-10.98365
4,frozen2022_parameters,10,carried_book,2026-07-31,2026-08-03,2026-09-25,38,19.70500,15.84322,3.33363,-11.82787,-10.98365
5,frozen2022_parameters,10,cash_at_first_trade,2026-07-31,2026-08-03,2026-09-25,38,15.38817,15.72750,-0.29321,-11.82786,-10.98365
6,frozen2022_parameters,25,carried_book,2026-07-31,2026-08-03,2026-09-25,38,19.51102,15.84322,3.16618,-11.86588,-10.98365
7,frozen2022_parameters,25,cash_at_first_trade,2026-07-31,2026-08-03,2026-09-25,38,15.07222,15.55434,-0.41723,-11.86587,-10.98365


Original SIGNAL window: [{'start_signal': '2026-08-03', 'last_exit': '2026-09-25', 'sessions': 37}]
Clarified TRADE window: [{'first_signal': '2026-07-31', 'first_trade': '2026-08-03', 'last_exit': '2026-09-25', 'sessions': 38}]


In [3]:
out=Path('/tmp')
table.to_csv(out/'benchmark_anchor_user_aug3.csv',index=False)
payload={'source_recorder_id':summary['recorder_id'],'parent_card':'research/benchmark_anchor_utility_20261002','policy':'price','historical_qualification_pass':False,'filter_column':'trade_date','filter_start':'2026-08-03','as_of':'2026-09-25','endpoint_mark':'OPEN','core_paths_modified':False,'trained_models_changed':False,'model_fits_performed':False,'scope':'fixed saved models; date-origin clarification only; no mode/parameter selection','source_artifacts':['inputs/diagnostic_frames.pkl','models/selected_annual_refits.pkl','analysis/portfolio_replays.pkl'],'source_hashes':summary['source_sha256'],'metrics':table.to_dict('records'),'limitations':['No new forward evidence; all periods already reviewed.','Carried-book and CASH-start are different estimands and must remain labelled.','This Sep25 OPEN endpoint must not be directly compared with previous Sep30 CLOSE results.','Historical source qualification is not live execution evidence.']}
(out/'benchmark_anchor_user_aug3.json').write_text(json.dumps(payload,indent=2,default=str,allow_nan=False))
with (out/'benchmark_anchor_user_aug3_paths.pkl').open('wb') as handle:pickle.dump({'source_recorder_id':summary['recorder_id'],'paths':new_paths},handle)
assert len(new_paths)==8
print('Saved temporary notebook-ready CSV/JSON and full paths for later Recorder archiving; no repository files changed.')

Saved temporary notebook-ready CSV/JSON and full paths for later Recorder archiving; no repository files changed.


In [4]:
lines=['## 中文结论','',f'按实际交易日8月3日起算，首信号日为{table.first_signal.iloc[0]}，首交易日为{table.first_trade.iloc[0]}，终点为{table.last_exit.iloc[0]}开盘，共{int(table.sessions.iloc[0])}个持有区间。旧表按信号日8月3日起算，首交易日8月4日，少一个区间；两表分别保留。','']
for row in table.loc[table.cost_bps.eq(10)].itertuples():
    lines.append(f'- {row.mode} / {row.origin}：净收益{row.return_pct:.3f}%，SOXX{row.benchmark_return_pct:.3f}%，相对财富变化{row.relative_return_pct:+.3f}%，最大回撤{row.max_drawdown_pct:.3f}%（SOXX{row.benchmark_max_drawdown_pct:.3f}%）。')
lines+=['','25bp结果与全部日期、期初持仓、成本见上表及JSON。本次仅修正用户窗口的筛选日期口径，没有训练或选择新策略。两种登记模式均应完整报告，不能事后挑选表现较好的模式。','', '该模型原历史资格仍未通过，生产不变；这些历史诊断不代表新的独立前向超额。数据只到9月25日开盘，不包含9月30日收盘，不能与旧9月30日结果直接比较。']
display(Markdown(chr(10).join(lines)))

## 中文结论

按实际交易日8月3日起算，首信号日为2026-07-31，首交易日为2026-08-03，终点为2026-09-25开盘，共38个持有区间。旧表按信号日8月3日起算，首交易日8月4日，少一个区间；两表分别保留。

- annual_refit / carried_book：净收益17.197%，SOXX15.843%，相对财富变化+1.169%，最大回撤-12.260%（SOXX-10.984%）。
- annual_refit / cash_at_first_trade：净收益12.846%，SOXX15.727%，相对财富变化-2.490%，最大回撤-12.260%（SOXX-10.984%）。
- frozen2022_parameters / carried_book：净收益19.705%，SOXX15.843%，相对财富变化+3.334%，最大回撤-11.828%（SOXX-10.984%）。
- frozen2022_parameters / cash_at_first_trade：净收益15.388%，SOXX15.727%，相对财富变化-0.293%，最大回撤-11.828%（SOXX-10.984%）。

25bp结果与全部日期、期初持仓、成本见上表及JSON。本次仅修正用户窗口的筛选日期口径，没有训练或选择新策略。两种登记模式均应完整报告，不能事后挑选表现较好的模式。

该模型原历史资格仍未通过，生产不变；这些历史诊断不代表新的独立前向超额。数据只到9月25日开盘，不包含9月30日收盘，不能与旧9月30日结果直接比较。